# 纯torch代码实现默认参数下的Embedding全过程


In [1]:
import json

import torch
# 加载权重文件
from safetensors.torch import load_file
from torch import nn


In [2]:
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

texts = ["Hello Word, a test sentence"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 从前一步得到的结果
token_ids = [9707, 9322, 11, 264, 1273, 11652, 151643]

vocab_size = 151669
# 查询得知

hidden_size = 1024
padding_idx = 151643

In [3]:
# 加载配置文件
config_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/config.json"

with open(config_file, "r", encoding="utf-8") as f:
    config = json.load(f)
config

{'architectures': ['Qwen3ForCausalLM'],
 'attention_bias': False,
 'attention_dropout': 0.0,
 'bos_token_id': 151643,
 'eos_token_id': 151643,
 'head_dim': 128,
 'hidden_act': 'silu',
 'hidden_size': 1024,
 'initializer_range': 0.02,
 'intermediate_size': 3072,
 'max_position_embeddings': 32768,
 'max_window_layers': 28,
 'model_type': 'qwen3',
 'num_attention_heads': 16,
 'num_hidden_layers': 28,
 'num_key_value_heads': 8,
 'rms_norm_eps': 1e-06,
 'rope_scaling': None,
 'rope_theta': 1000000,
 'sliding_window': None,
 'tie_word_embeddings': True,
 'torch_dtype': 'bfloat16',
 'transformers_version': '4.51.3',
 'use_cache': True,
 'use_sliding_window': False,
 'vocab_size': 151669}

# 2. Embedding lookup
根据 token id 在 embedding 矩阵中索引对应向量

In [4]:
# nn.Embedding
embed_tokens = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=hidden_size,
    padding_idx=padding_idx,
)
embed_tokens

Embedding(151669, 1024, padding_idx=151643)

In [5]:
token_ids = torch.tensor(token_ids, dtype=torch.long)
token_ids


tensor([  9707,   9322,     11,    264,   1273,  11652, 151643])

# 模型权重加载
后面就需要用到模型权重了。这里模型权重处理

In [6]:
model_file = "/Users/pengjunzhe/.cache/huggingface/hub/models--Qwen--Qwen3-Embedding-0.6B/snapshots/c54f2e6e80b2d7b7de06f51cec4959f6b3e03418/model.safetensors"

# 读取文件
state_dict = load_file(model_file, device="cpu")  # 返回 dict: key -> torch.Tensor

# 查看有哪些 key
print(list(state_dict.keys()))

['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_norm.weight', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.weight', 'layers.10.input_layernorm.weight', 'layers.10.mlp.down_proj.weight', 'layers.10.mlp.gate_proj.weight', 'layers.10.mlp.up_proj.weight', 'layers.10.post_attention_layernorm.weight', 'layers.10.

In [7]:
# 大致看起来没有异常的键，但还是整理一下
# 处理键名（如果需要）
# 例如：移除 "model." 前缀
new_state_dict = {}
for key, value in state_dict.items():
    new_key = key
    # 根据你的模型结构调整
    # new_key = key.replace("model.", "")
    new_state_dict[new_key] = value
print(list(new_state_dict.keys()))

['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_norm.weight', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_norm.weight', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_norm.weight', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_norm.weight', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.weight', 'layers.10.input_layernorm.weight', 'layers.10.mlp.down_proj.weight', 'layers.10.mlp.gate_proj.weight', 'layers.10.mlp.up_proj.weight', 'layers.10.post_attention_layernorm.weight', 'layers.10.

In [8]:
# 提取 embedding 权重
embedding_weights = state_dict['embed_tokens.weight']
embedding_weights.shape

torch.Size([151669, 1024])

In [9]:
# 直接加载没有问题
embed_tokens.weight.data.copy_(embedding_weights)

tensor([[-0.0031,  0.0327, -0.0703,  ...,  0.0138, -0.0144,  0.0128],
        [ 0.0303,  0.0244, -0.0613,  ..., -0.0031, -0.0374,  0.0077],
        [ 0.0292,  0.0322, -0.0223,  ..., -0.0095,  0.0025,  0.0256],
        ...,
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026],
        [ 0.0031, -0.1064,  0.0245,  ...,  0.0028, -0.0055, -0.0026]])

In [10]:
# 使用 nn.Embedding 查表
inputs_embeds = embed_tokens(token_ids)  # shape: [7, 1024]
print(inputs_embeds.shape)
print(inputs_embeds[0])

torch.Size([7, 1024])
tensor([ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
       grad_fn=<SelectBackward0>)



# 位置信息（Positional Encoding）——告诉模型“顺序”

In [11]:
from transformers import DynamicCache

# kv缓存用的
past_key_values = DynamicCache()
print(f"past_key_values: {past_key_values}")

# past_seen_tokens 表示已处理过的 token 数量，用于计算当前输入在序列中的绝对位置。
past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
print(f"past_seen_tokens: {past_seen_tokens}")

# cache_position 表示当前输入在 KV 缓存中的位置索引，用于增量生成。
# 首次前向：past_seen_tokens = 0，cache_position = [0, 1, 2, ...]
# 后续生成：past_seen_tokens = 已生成长度，cache_position = [已生成长度, 已生成长度+1, ...]
cache_position = torch.arange(
    past_seen_tokens, past_seen_tokens + inputs_embeds.shape[0], device=inputs_embeds.device
)
print(f"cache_position: {cache_position}")

# position_ids 是每个 token 的绝对位置索引，用于位置编码（如 RoPE）。
# 形状：[1, seq_len]
# 内容：[[0, 1, 2, ...]] 或 [[past_seen_tokens, past_seen_tokens+1, ...]]
# position_ids = cache_position.unsqueeze(-1)
# 这里不考虑batch，也就不unsqueeze了
position_ids = cache_position
print(f"position_ids: {position_ids}")

# 计算 attention_mask
# 到这里padding_size是None
attention_mask = torch.ones_like(token_ids)
print(f"attention_mask: {attention_mask}")

past_key_values: DynamicCache(layers=[])
past_seen_tokens: 0
cache_position: tensor([0, 1, 2, 3, 4, 5, 6])
position_ids: tensor([0, 1, 2, 3, 4, 5, 6])
attention_mask: tensor([1, 1, 1, 1, 1, 1, 1])


# 构建推理模型

In [12]:
# 初始的隐藏层就是上面Embedding lookup的结果
hidden_states = inputs_embeds
print(hidden_states[0])

tensor([ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
       grad_fn=<SelectBackward0>)


## 定义RoPE
旋转位置编码（Rotary Position Embedding, RoPE），这是一种将位置信息编码到 Transformer 模型中的方法。
与传统的绝对位置编码不同，RoPE 通过旋转操作将相对位置信息直接嵌入到 query 和 key 向量中。
论文地址：https://arxiv.org/abs/2104.09864

注意和transformer原始论文Attention is all you need中固定正弦/余弦位置编码（sinusoidal positional encoding, Sine-PE）区别和优点

In [13]:
# 定义RoPE
from rotary_embedding import RotaryEmbedding
rotary_emb = RotaryEmbedding()
position_embeddings = rotary_emb(hidden_states, position_ids=position_ids)
print(f"position_embeddings: {position_embeddings[0][1]}")

position_embeddings: tensor([0.5403, 0.6925, 0.7965, 0.8662, 0.9124, 0.9428, 0.9627, 0.9758, 0.9842,
        0.9897, 0.9933, 0.9957, 0.9972, 0.9982, 0.9988, 0.9992, 0.9995, 0.9997,
        0.9998, 0.9999, 0.9999, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 0.5403, 0.6925, 0.7965, 0.8662, 0.9124, 0.9428, 0.9627, 0.9758,
        0.9842, 0.9897, 0.9933, 0.9957, 0.9972, 0.9982, 0.9988, 0.9992, 0.9995,
        0.9997, 0.9998, 0.9999, 0.9999, 0.9999, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0

## 开始逐层构建推理框架
### Layer层构成


### RMSNorm
RMSNorm（Root Mean Square Layer Normalization） 是一种归一化技术，相当于简化版的 LayerNorm。

他的前向计算步骤：
1. 计算输入的平方的均值（方差）
2. 用均方根的倒数来缩放输入
3. 乘以可学习的权重参数

在 Attention 中使用 RMSNorm 的好处
- 稳定注意力分数的计算：对 Q 和 K 进行归一化后，它们的点积（注意力分数）会更加稳定，防止数值过大或过小
- 提高训练稳定性：避免梯度爆炸或消失，特别是在深层模型中
- 计算效率更高：相比标准 LayerNorm，RMSNorm 不需要计算和减去均值，只需要计算均方根，运算更简单
- 适配大模型训练：在大规模语言模型（如 Qwen3）中，这种设计已被证明能提升性能和稳定性
- 改善注意力质量：归一化后的 Q 和 K 有助于注意力机制更好地捕捉相关性，而不受向量幅度的影响
- 这是现代 Transformer 架构（如 LLaMA、Qwen 等）的一个重要改进，相比原始的 Transformer 设计更加高效和稳定。



In [14]:
from tests.learn_embedding.decode_layer import DecoderLayer
from tests.learn_embedding.rms_norm import RMSNorm
from tests.learn_embedding.rotary_embedding import RotaryEmbedding
# 根据模型配置构建transformer深度神经网络
layers = nn.ModuleList(
    [DecoderLayer(layer_idx) for layer_idx in range(config["num_hidden_layers"])]
)
norm = RMSNorm(
    config["hidden_size"],
    eps=config["rms_norm_eps"],
)
rotary_emb = RotaryEmbedding()

In [15]:
print(layers)
print(norm)
print(rotary_emb)

ModuleList(
  (0-27): 28 x DecoderLayer(
    (self_attn): Attention(
      (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
      (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
      (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
      (q_norm): RMSNorm((128,), eps=1e-06)
      (k_norm): RMSNorm((128,), eps=1e-06)
    )
    (mlp): MLP(
      (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
      (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
      (act_fn): SiLU()
    )
    (input_layernorm): RMSNorm((1024,), eps=1e-06)
    (post_attention_layernorm): RMSNorm((1024,), eps=1e-06)
  )
)
RMSNorm((1024,), eps=1e-06)
RotaryEmbedding()


In [16]:
from embedding_model import EmbeddingModel
model = EmbeddingModel(
    embed_tokens=embed_tokens,
    layers=layers,
    norm=norm,
   rotary_emb=rotary_emb,
)
missing, unexpected = model.load_state_dict(state_dict,strict=True)
print("==> 加载完成")
if missing:
    print("missing keys:", missing[:20])
if unexpected:
    print("unexpected keys:", unexpected[:20])


==> 加载完成


# 推理！推理！推理！

In [17]:
print(hidden_states)

tensor([[ 0.0027, -0.0106,  0.0182,  ..., -0.0143, -0.0408,  0.0062],
        [ 0.0031,  0.0045, -0.0206,  ..., -0.0312, -0.0231,  0.0123],
        [-0.0201,  0.0503, -0.0757,  ..., -0.0210,  0.0055, -0.0255],
        ...,
        [-0.0214, -0.0391, -0.0535,  ...,  0.0430, -0.0386, -0.0222],
        [ 0.0170, -0.0674,  0.0131,  ...,  0.0086,  0.0210, -0.0094],
        [-0.0043,  0.0435, -0.0334,  ..., -0.0039,  0.0449,  0.0320]],
       grad_fn=<EmbeddingBackward0>)


In [18]:
for layer in layers:
    hidden_states = layer(hidden_states, position_embeddings=position_embeddings)
    print(hidden_states)
hidden_states

tensor([[-0.3820,  0.2517,  0.5030,  ..., -0.0908,  0.3696, -0.0876],
        [-0.0136,  0.1875,  0.2521,  ..., -0.0176, -0.2009, -0.3630],
        [-0.4227,  0.5275, -0.0958,  ..., -0.0058, -0.3467,  0.0875],
        ...,
        [-0.2078,  0.0492, -0.2577,  ..., -0.3179, -0.2039,  0.1548],
        [-0.0898, -0.0467,  0.1042,  ..., -0.1682,  0.2290, -0.3348],
        [-0.2057,  0.2684, -0.2771,  ...,  0.2716, -0.4624, -0.1887]],
       grad_fn=<AddBackward0>)
tensor([[-0.6972,  0.1967,  0.6602,  ...,  0.0295,  0.0164, -0.1956],
        [-0.7009,  0.5725,  0.0444,  ..., -0.7410, -0.6995,  0.2090],
        [-0.7431,  0.7465,  0.0624,  ..., -0.1441,  0.1056,  0.8991],
        ...,
        [-0.2035,  0.6617, -0.2795,  ..., -0.3577,  0.0948,  0.4737],
        [ 0.3608,  0.3927, -0.2723,  ...,  0.8100,  1.4274, -0.4015],
        [-0.6725,  0.7908, -1.1137,  ..., -0.4735,  0.0584, -0.7420]],
       grad_fn=<AddBackward0>)
tensor([[-0.7447,  0.5584,  0.6612,  ...,  0.4020,  0.4013, -0.0728],


tensor([[  -3.7108,   -6.6081,   36.6548,  ...,    2.1135,   -5.7093,
           19.7635],
        [  18.8031,   19.8499,  -99.1394,  ...,  -52.6457,    7.3374,
           14.5165],
        [ -21.6979,    7.5989,  -16.8371,  ...,   -0.8329,   66.9671,
          -17.0787],
        ...,
        [ -25.9289,  -17.4127, -233.4510,  ...,  -41.8504,  -27.9556,
          -61.8764],
        [  -2.9542,    5.3450,  -14.6033,  ...,    6.7941,   21.2113,
          -25.1197],
        [  -0.2637,  -38.6826, -124.2231,  ...,    8.6489,  -54.9741,
          -14.3962]], grad_fn=<AddBackward0>)

In [19]:
# 掩码配置
mask_kwargs = {
    "config": None,
    "input_embeds": inputs_embeds,
    "attention_mask": attention_mask,
    "cache_position": cache_position,
    "past_key_values": past_key_values,
    "position_ids": position_ids,
}

In [20]:
# Create the masks
causal_mask_mapping = {
    "full_attention": None
}